# Training / Validation / Testing splitting

In [ ]:
# general
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import pandas as pd
import numpy as np

from src.utils import load_config
cfg = load_config()

In [ ]:
# metadata
metadata = pd.read_csv(Path(cfg["dir"]["pipeline"]) / "lstm" / "data" / "attributes" / "basin_attributes.csv", index_col=0)

metadata_target = metadata.loc[~metadata.index.str.match(r'^[PX]\d+')]  # Start with X or P followed by digits
metadata  = metadata.loc[metadata.index.str.match(r'^[PX]\d+')]   # Everything else (true)

In [ ]:
# remove training basins with less than 50% of streamflow data
data = pd.read_csv(cfg["dir"]["andeangc"] + "/dataset/AndeanGC_data_1950_2024_qc.csv", index_col=0, parse_dates=True)
data = data.loc["2000-01-01":cfg["time"]["ref_end"]]  #TODO check 1994-2000

data_available = data.notnull().sum() / data.shape[0]
data = data.loc[:, data_available > cfg["hydro"]["min_data"]] 
metadata = metadata.loc[data.columns]

# remove training basins with: cropland cover > 20%, urban cover > 5% and big dams (> 100 hm3)
metadata = metadata[metadata.cropland_cover < 20]
metadata = metadata[metadata.urban_cover < 5]
metadata = metadata[metadata.dams_count == 0]

In [ ]:
# visual inspection of the training basins
to_remove = ["X00001870", "X00002218", "X00002239", "X03802001", 
             "X03022001", "X01020003", "X03814003", "X03820003",
             "X08376001", "X00002282", "X11536001"] 

metadata = metadata[~metadata.index.isin(to_remove)]
print("Total number of basins:", len(metadata))

In [ ]:
# attributes selection
static_attributes = [
    "basin_cenlat", "basin_cenlon",                              # location
    "basin_area", "slope_mean", "aspect_mean", 'elev_std',       # topography
    "shrubs_veg_cover", "forest_cover", "herbaceous_veg_cover",  # land cover
    "glacier_area_RGI60", "glacier_dhdt",                        # glacier
    'prcp_mean_ERA5', 'prcp_pci_ERA5', 'ev_mean_ERA5',           # climate
    'solid_prcp_ERA5', 'frac_snow_ERA5']                         # climate

metadata = metadata[static_attributes]

# Standardize data using joint statistics to ensure consistent scaling
scaler = StandardScaler()
scaler.fit(metadata)

metadata = pd.DataFrame(
    scaler.transform(metadata),
    index=metadata.index,
    columns=static_attributes)

In [ ]:
# Optimize train/validation/test split using Wasserstein distance
n_test = int(len(metadata) * 0.10)
n_val = int(len(metadata) * 0.10)
n_reps = 30
best_score = -np.inf
best_split = None

for seed in range(n_reps):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(metadata))
    
    # Split indices: 10% test, 10% validation, 80% train
    test_idx = perm[:n_test]
    val_idx = perm[n_test:n_test + n_val]
    train_idx = perm[n_test + n_val:]
    
    # Subsample training to match val/test size (for fair distance comparison)
    train_sample_idx = rng.choice(train_idx, size=n_val, replace=False)
    
    # Get data subsets
    train_sample = metadata.iloc[train_sample_idx]
    val_data = metadata.iloc[val_idx]
    test_data = metadata.iloc[test_idx]
    
    # Calculate Wasserstein distances between all pairs
    distances = []
    for col in static_attributes:
        d1 = wasserstein_distance(train_sample[col], val_data[col])
        d2 = wasserstein_distance(val_data[col], test_data[col])
        d3 = wasserstein_distance(test_data[col], train_sample[col])
        distances.append(d1 + d2 + d3)
    
    score = np.mean(distances)
    
    # Maximize distance for better out-of-sample generalization
    if score > best_score:
        best_score = score
        best_split = (train_idx, val_idx, test_idx)

# Extract basin IDs
train_basins = metadata.index[best_split[0]].tolist()
val_basins = metadata.index[best_split[1]].tolist()
test_basins = metadata.index[best_split[2]].tolist()

print(f"Best Wasserstein distance: {best_score:.4f}")
print(f"Split sizes - Train: {len(train_basins)}, Val: {len(val_basins)}, Test: {len(test_basins)}")

In [ ]:
# create basin lists 
with open(Path("modelling") / "basins_training.txt", "w") as file: # train 
    for x in train_basins:
        file.write(x + "\n")

with open(Path("modelling") / "basins_validation.txt", "w") as file: # validation
    for x in val_basins:
        file.write(x + "\n")

with open(Path("modelling") / "basins_test.txt", "w") as file: # testing + target basins
    for x in test_basins:
        file.write(x + "\n")

with open(Path("modelling") / "basins_test_target.txt", "w") as file: # testing + target basins
    for x in metadata_target.index.tolist():
        file.write(x + "\n")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# Use the selected split
train_sel = metadata.loc[train_basins]
val_sel   = metadata.loc[val_basins]
target_sel = metadata.loc[test_basins]  # already filtered to static_attributes

# Plot overlapping histograms for each static attribute with shared bins, showing percentage
n_cols = 4
n_rows = int(np.ceil(len(static_attributes) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, attr in enumerate(static_attributes):
    ax = axes[i]
    # Common bins across Train/Val/Target
    combined = pd.concat([train_sel[attr], val_sel[attr], target_sel[attr]])
    bins = np.histogram_bin_edges(combined.dropna(), bins=30)

    ax.hist(train_sel[attr].dropna(), bins=bins, alpha=0.55, label='Train',
            color='tab:blue', edgecolor='black', density=True)
    ax.hist(val_sel[attr].dropna(), bins=bins, alpha=0.55, label='Val',
            color='tab:green', edgecolor='black', density=True)
    ax.hist(target_sel[attr].dropna(), bins=bins, alpha=0.55, label='Target',
            color='tab:orange', edgecolor='black', density=True)

    ax.set_title(attr)
    ax.set_xlabel('Value')
    ax.set_ylabel('Percentage')
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    ax.legend()

# Hide any unused subplots
for i in range(len(static_attributes), len(axes)):
    axes[i].set_visible(False)

plt.tight_layout()